# Step 2: TutorAgent 기초 구축

LangGraph Multi-Agent + ReAct 패턴으로 기본 그래프를 구현합니다.

## 아키텍쳐

```
START → supervisor_agent → (transfer_to_agent) → search_agent
                                                       ↓
                                                  quiz_agent
                                                       ↓
                                                  qna_agent
                                                       ↓
                                                  tutor_agent
                                                       ↓
                                                      END
```

### 핵심 패턴
- **create_agent**: 각 에이전트가 도구를 사용하는 ReAct 루프
- **transfer_to_agent**: Command(goto=...) 기반 에이전트 간 전환
- **Supervisor Router**: Supervisor가 사용자 입력을 분석하여 전문 에이전트로 라우팅

## 1. 환경 설정

In [1]:
from pathlib import Path
import sys
import os
import importlib
from dotenv import load_dotenv

# Ensure local package imports work even when notebook cwd is ./notebooks
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

# 모듈 캐시 제거 (소스 변경 시 최신 코드 반영)
for mod_name in [m for m in sys.modules if m.startswith("tutor_agent")]:
    del sys.modules[mod_name]

print(f"API Key 설정: {'OK' if os.environ.get('GOOGLE_API_KEY') else 'MISSING'}")

API Key 설정: OK


## 2. 그래프 빌드 및 시각화

In [2]:
from tutor_agent.agents.graph import build_graph

graph = build_graph()

# Mermaid 다이어그램
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor_agent(supervisor_agent)
	search_agent(search_agent)
	quiz_agent(quiz_agent)
	qna_agent(qna_agent)
	tutor_agent(tutor_agent)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> qna_agent;
	__start__ -.-> quiz_agent;
	__start__ -.-> search_agent;
	__start__ -.-> supervisor_agent;
	__start__ -.-> tutor_agent;
	supervisor_agent -.-> qna_agent;
	supervisor_agent -.-> quiz_agent;
	supervisor_agent -.-> search_agent;
	supervisor_agent -.-> tutor_agent;
	qna_agent --> __end__;
	quiz_agent --> __end__;
	search_agent --> __end__;
	tutor_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. State 구조 확인

In [3]:
from tutor_agent.agents.state import TutorAgentState

print("TutorAgentState 필드:")
for name, field_type in TutorAgentState.__annotations__.items():
    print(f"  {name}: {field_type}")

TutorAgentState 필드:
  messages: ForwardRef('Annotated[list[AnyMessage], add_messages]', module='langgraph.graph.message')
  current_agent: <class 'str'>
  command_type: <class 'str'>
  user_message: <class 'str'>
  session_active: <class 'bool'>
  session_subject: <class 'str'>
  memos: list[str]
  material: <class 'str'>
  material_valid: <class 'bool'>
  quiz: <class 'dict'>
  quiz_valid: <class 'bool'>
  generation_attempts: <class 'int'>
  answer: <class 'str'>
  error: <class 'str'>


## 4. 에이전트 구조 확인

In [4]:
from tutor_agent.agents.graph import AGENT_NODES

print("등록된 에이전트:")
for name in AGENT_NODES:
    print(f"  - {name}")

print("\n도구 목록:")
from tutor_agent.agents.tools.shared_tools import transfer_to_agent
from tutor_agent.agents.tools.tutor_tools import search_material, get_study_memos

for tool in [transfer_to_agent, search_material, get_study_memos]:
    print(f"  - {tool.name}: {tool.description[:50]}...")

등록된 에이전트:
  - supervisor_agent
  - search_agent
  - quiz_agent
  - qna_agent
  - tutor_agent

도구 목록:
  - transfer_to_agent: 다른 전문 에이전트로 전환합니다.

Args:
    agent_name: 전환할 에이전트...
  - search_material: 강의 자료를 검색합니다.

Args:
    subject: 과목명과 주차 (예: '양택풍...
  - get_study_memos: 해당 과목의 학습 메모를 조회합니다.

Args:
    subject: 과목명과 주차 (...


## 5. 그래프 실행 테스트

Supervisor가 사용자 입력을 분석하여 적절한 에이전트로 라우팅합니다.

> **참고**: 실제 Gemini API를 호출합니다. GOOGLE_API_KEY가 설정되어 있어야 합니다.

In [5]:
from langchain_core.messages import HumanMessage

# 퀴즈 생성 요청
result = graph.invoke({
    "messages": [HumanMessage(content="양택풍수론 4주차 퀴즈 내줘")],
})

# 결과 확인
print("=== 실행 결과 ===")
for msg in result["messages"]:
    role = msg.__class__.__name__
    content = msg.content[:200] if isinstance(msg.content, str) else str(msg.content)[:200]
    print(f"\n[{role}] {content}")

[TOOL] transfer_to_agent — goto=quiz_agent
[TOOL] search_material — subject=양택풍수론 4주차
[TOOL] get_study_memos — subject=양택풍수론 4주차
=== 실행 결과 ===

[HumanMessage] 양택풍수론 4주차 퀴즈 내줘

[AIMessage] 

[ToolMessage] {"subject": "양택풍수론 4주차", "content": "[양택풍수론 4주차] 강의 자료 내용\n\n1. 핵심 개념: 양택풍수론 4주차의 기본 원리와 역사적 배경\n2. 주요 용어: 용어A(정의), 용어B(정의), 용어C(정의)\n3. 중요 내용: 이론적 프레임워크와 실제 적용 사례\n4. 심화 내용: 최신 연구 동향과 학술적 논의\n5. 실습 요

[AIMessage] 

[ToolMessage] {"subject": "양택풍수론 4주차", "memos": ["한옥 지붕의 팔작지붕과 우진각지붕 차이 중요", "기단의 삼분할 개념 시험에 나올듯"], "count": 2}

[AIMessage] [{'type': 'text', 'text': '```json\n{\n  "title": "양택풍수론 4주차 퀴즈",\n  "questions": [\n    {\n      "type": "multiple_choice",\n      "question": "양택풍수론 4주차의 핵심 개념 중 하나로, 주택의 길흉화복을 결정하는 가장 기본적인 원리는 무엇인가


In [6]:
# 질문 답변 요청
result2 = graph.invoke({
    "messages": [HumanMessage(content="풍수에서 생기란 무엇인가요?")],
})

print("=== 질문 답변 결과 ===")
for msg in result2["messages"]:
    role = msg.__class__.__name__
    content = msg.content[:200] if isinstance(msg.content, str) else str(msg.content)[:200]
    print(f"\n[{role}] {content}")

[TOOL] transfer_to_agent — goto=qna_agent
[TOOL] search_material — subject=풍수 생기
=== 질문 답변 결과 ===

[HumanMessage] 풍수에서 생기란 무엇인가요?

[AIMessage] 

[ToolMessage] {"subject": "풍수 생기", "content": "[풍수 생기] 강의 자료 내용\n\n1. 핵심 개념: 풍수 생기의 기본 원리와 역사적 배경\n2. 주요 용어: 용어A(정의), 용어B(정의), 용어C(정의)\n3. 중요 내용: 이론적 프레임워크와 실제 적용 사례\n4. 심화 내용: 최신 연구 동향과 학술적 논의\n5. 실습 요소: 현장 적용 방법론

[AIMessage] [{'type': 'text', 'text': "강의 자료에서 '풍수 생기'에 대한 직접적인 정의는 확인할 수 없습니다. 자료에는 '풍수 생기의 기본 원리와 역사적 배경', '주요 용어' 등의 목차가 제시되어 있지만, '생기' 자체의 구체적인 설명은 포함되어 있지 않습니다.", 'extras': {'signature': 'Co0LAb4+9vs0khmhG5V


## 요약

### 구현된 패턴

| 패턴 | 구현 |
|------|------|
| **create_agent** | 모든 에이전트가 ReAct 루프로 도구 사용 |
| **transfer_to_agent** | Command(goto=..., graph=PARENT)로 에이전트 전환 |
| **Supervisor Router** | supervisor_agent가 자연어 분석 후 라우팅 |
| **@tool 데코레이터** | 도메인 도구 정의 (dummy) |
| **프롬프트 모듈화** | TRANSFER_SUFFIX 공통 접미사 |

### 파일 구조

```
tutor_agent/agents/
├── __init__.py              # 모델 설정, get_model()
├── state.py                 # TutorAgentState(MessagesState)
├── graph.py                 # StateGraph 빌드 (Supervisor Router)
├── prompts.py               # TRANSFER_SUFFIX
├── supervisor_agent.py      # 라우팅 (transfer_to_agent 도구)
├── search_agent.py          # 자료 검색 (search_material 도구)
├── quiz_agent.py            # 퀴즈 생성 (search_material + get_study_memos)
├── qna_agent.py             # Q&A 답변 (search_material 도구)
├── tutor_agent.py           # 1:1 과외 (에이전트 주도 학습)
└── tools/
    ├── shared_tools.py      # transfer_to_agent (Command)
    └── tutor_tools.py       # search_material, get_study_memos (dummy)
```

### 다음 단계
- Gemini File Search 실제 연동 (tutor_tools.py)
- 학습 세션 관리 (/start, /done, /memo)
- Slack 플랫폼 어댑터 연동